In [1]:
#download mediapipe face landmarker 
import requests

url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
response = requests.get(url)
if response.status_code ==200:
    with open("face_landmarker.task", "wb") as f:
        f.write(response.content)
    print("Downloaded face_landmarker.task successfully.")
else:
    print(f"Failed to download face_landmarker.task. Status code: {response.status_code}")

Downloaded face_landmarker.task successfully.


In [2]:
import os

data_dir = os.path.join("../dataset","1","data")

In [3]:

import cv2
import numpy as np
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os


MODEL_PATH = "face_landmarker.task"
LIP_LANDMARKS = [
    61,146,91,181,84,17,314,405,
    321,375,291,308,324,318,402,317,
    14,87,178,88,95
]

def extract_roi(image,landmarker):
   
    h, w, _ = image.shape
    # Convert to MediaPipe image
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)  
    result = landmarker.detect(mp_image)

    if not result.face_landmarks:
        raise ValueError("No face detected")

    landmarks = result.face_landmarks[0]

    # Extract lip points
    pts = []
    for idx in LIP_LANDMARKS:
        lm = landmarks[idx]
        x, y = int(lm.x * w), int(lm.y * h)
        pts.append((x, y))


    pts = np.array(pts)

    # Bounding box
    x_min, x_max = np.min(pts[:,0]), np.max(pts[:,0])
    y_min, y_max = np.min(pts[:,1]), np.max(pts[:,1])

    margin = 10
    x_min = max(0, x_min - margin)
    y_min = max(0, y_min - margin)
    x_max = min(w, x_max + margin)
    y_max = min(h, y_max + margin)

    lip_roi = image[y_min:y_max, x_min:x_max]

    if lip_roi.size == 0:
        raise ValueError("Empty ROI")
       
    return lip_roi 

In [ ]:

roi_failed_videos =[]
roi_failed_count = 0 
BaseOptions = python.BaseOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
VisionRunningMode = vision.RunningMode


options = FaceLandmarkerOptions(
                base_options=BaseOptions(model_asset_path=MODEL_PATH),
                running_mode=VisionRunningMode.IMAGE,num_faces=1)

landmarker = FaceLandmarker.create_from_options(options)
for speaker in os.listdir(data_dir):
    print("Processing speaker:", speaker)
    speaker_dir = os.path.join(data_dir, speaker)
    speaker_dir_list = os.listdir(speaker_dir)
    for file in speaker_dir_list:
        if file.endswith(".mpg"):
            
            updated_video =[]
            video_path = os.path.join(speaker_dir, file)
            #check if the roi has already been extracted on previous runs
            if os.path.exists(os.path.join(speaker_dir, file.replace(".mpg", "_lip_roi.npy"))):
                continue
            cap = cv2.VideoCapture(video_path)
            # Create detector
           
            while True: 
                
                _, frame = cap.read()
                if frame is None:
                    break 
                # Extract ROI 
                try: 
                    lip_roi = extract_roi(frame,landmarker)
                    lip_roi = cv2.resize(lip_roi, (128, 128))
                    lip_roi = cv2.cvtColor(lip_roi, cv2.COLOR_BGR2GRAY)
                    updated_video.append(lip_roi)
                    
                except ValueError as e:
                    roi_failed_videos.append(video_path)
                    roi_failed_count += 1
                    print(roi_failed_count)
                    
            cap.release()
            with open(os.path.join(speaker_dir, file.replace(".mpg", "_lip_roi.npy")), "wb") as f:
                np.save(f, np.array(updated_video))


I0000 00:00:1784538651.359118    8197 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1784538651.419794   14913 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: llvmpipe (LLVM 20.1.2, 256 bits)
W0000 00:00:1784538651.438565    8197 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1784538651.460678   14917 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784538651.476503   14916 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use mess

Processing speaker: s25_processed
Processing speaker: s26_processed
Processing speaker: s4_processed
Processing speaker: s14_processed
Processing speaker: s27_processed
Processing speaker: s22_processed
Processing speaker: s20_processed
Processing speaker: s31_processed
Processing speaker: s30_processed
Processing speaker: s2_processed
Processing speaker: s17_processed
Processing speaker: s13_processed
Processing speaker: s5_processed
Processing speaker: s1_processed
Processing speaker: s11_processed
Processing speaker: s32_processed


Processing speaker: s33_processed
